In [3]:
"""
rnn_models.py - Reto 9 (Data mining)
RNN en PyTorch que predice el payoff de la call asiatica sobre 3 activos.

Entrada: ../DATOS/TRANSFORMADOS/datos_sinteticos.npy  -> forma (100000, 21, 3), precios con S0 = 100.
"""
import os
import time
import numpy as np
import torch
import torch.nn as nn
from scipy.stats.mstats import gmean
from sklearn.model_selection import KFold


In [2]:
%pip install torch scikit-learn scipy

   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.1 MB 6.7 MB/s eta 0:00:19
   - -------------------------------------- 3.4/124.1 MB 11.3 MB/s eta 0:00:11
   - -------------------------------------- 6.0/124.1 MB 12.3 MB/s eta 0:00:10
   -- ------------------------------------- 7.1/124.1 MB 10.5 MB/s eta 0:00:12
   -- ------------------------------------- 8.4/124.1 MB 8.9 MB/s eta 0:00:13
   --- ------------------------------------ 10.7/124.1 MB 9.6 MB/s eta 0:00:12
   ---- ----------------------------------- 14.2/124.1 MB 10.6 MB/s eta 0:00:11
   ----- ---------------------------------- 18.1/124.1 MB 11.7 MB/s eta 0:00:10
   ------ --------------------------------- 21.5/124.1 MB 12.3 MB/s eta 0:00:09
   ------- -------------------------------- 23.6/124.1 MB 12.2 MB/s eta 0:00:09
   -------- ------------------------------- 25.4/124.1 MB 11.8 MB/s eta 0:00:09
   --------- ------------------------------ 28.8/124.1 MB

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
data = np.load(os.path.join('..', 'DATOS', 'TRANSFORMADOS', 'datos_sinteticos.npy'))

In [5]:

MU, TTM, N_PASOS, E = 0.01, 5, 20, 3.0
DT = TTM / N_PASOS

def payoff(X):
    """V_T = (suma de las 3 medias geometricas - E)^+"""
    return np.maximum(gmean(X, axis=1).sum(axis=1) - E, 0.0)


def prima(V_T, t_n=0):
    return np.exp(-MU * (N_PASOS - t_n) * DT) * V_T



In [6]:
class RNN(nn.Module):
    def __init__(self, cell="lstm", units=64, n_layers=1, dropout=0.1):
        super().__init__()
        Cell = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = Cell(3, units, n_layers, batch_first=True,
                        dropout=dropout if n_layers > 1 else 0.0)
        self.out = nn.Linear(units, 1)

    def forward(self, x):
        h, _ = self.rnn(x)
        return self.out(h[:, -1]).squeeze(-1)   # ultimo paso temporal


def tensor(a):
    return torch.tensor(a, dtype=torch.float32)


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(tensor(X)).numpy()


def train(model, X_tr, y_tr, X_val, y_val, epochs=60, lr=1e-3, bs=256, patience=8):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    Xt, yt = tensor(X_tr), tensor(y_tr)
    best, best_state, wait = np.inf, None, 0

    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            loss_fn(model(Xt[idx]), yt[idx]).backward()
            opt.step()

        val = np.mean((predict(model, X_val) - y_val) ** 2)
        if val < best:
            best, wait = val, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:     # early stopping
                break

    model.load_state_dict(best_state)
    return model



In [7]:
def metricas(y, y_pred):
    err = y_pred - y
    return {"RMSE": np.sqrt(np.mean(err ** 2)),
            "MAE": np.mean(np.abs(err)),
            "R2": 1 - np.sum(err ** 2) / np.sum((y - y.mean()) ** 2)}


In [8]:
def cross_val(cfg, X, y, k=5):
    resultados = []
    for tr, te in KFold(k, shuffle=True, random_state=42).split(X):
        n_val = int(0.15 * len(tr))
        va, tr = tr[:n_val], tr[n_val:]

        # escalado ajustado solo con train
        m, s = X[tr].mean((0, 1)), X[tr].std((0, 1))
        ys = y[tr].std()
        esc = lambda a: (a - m) / s

        torch.manual_seed(0)
        model = RNN(**cfg)
        train(model, esc(X[tr]), y[tr] / ys, esc(X[va]), y[va] / ys)
        resultados.append(metricas(y[te], predict(model, esc(X[te])) * ys))

    return {k_: (np.mean([r[k_] for r in resultados]),
                 np.std([r[k_] for r in resultados])) for k_ in resultados[0]}



In [9]:
if __name__ == "__main__":
    data = np.load(os.path.join('..', 'DATOS', 'TRANSFORMADOS', 'datos_sinteticos.npy'))
    X_ratio = data[:, 1:, :] / data[:, :1, :]     # (100000, 20, 3): quita t=0 y pasa a ratios S_j/S_0
    print(f"shape original: {data.shape} | shape ratios: {X_ratio.shape}")

    y = payoff(X_ratio)
    X = np.diff(np.log(X_ratio), axis=1, prepend=0.0)   # log-retornos por paso
    print(f"% payoffs = 0: {100 * np.mean(y == 0):.1f}% | prima media t0: {prima(y).mean():.4f}")

    configs = {
        "RNN":     dict(cell="rnn",  units=64, n_layers=1),
        "LSTM":    dict(cell="lstm", units=64, n_layers=1),
        "GRU":     dict(cell="gru",  units=64, n_layers=1),
        "LSTM_2L": dict(cell="lstm", units=64, n_layers=2),
    }
    for nombre, cfg in configs.items():
        res = cross_val(cfg, X, y)
        print(f"\n{nombre}")
        for m, (media, std) in res.items():
            print(f"  {m}: {media:.4f} +- {std:.4f}")

shape original: (100000, 21, 3) | shape ratios: (100000, 20, 3)
% payoffs = 0: 60.7% | prima media t0: 0.1523


KeyboardInterrupt: 